# Phase 5 (LSI): one Phase 1 model, full pipeline
Set `MODEL` in cell 2, then **Save Version -> Save & Run All** (runs with the tab closed; everything in /kaggle/working is
kept as the version's output). One notebook per model:

| notebook | MODEL | account | inputs | about |
|---|---|---|---|---|
| phase5-llama | `"llama"` | yours | checkpoints-models | 7-9 h (includes knowledge distillation) |
| phase5-decoder | `"decoder"` | yours (after llama) | checkpoints-models | 6-8 h |
| phase5-moe | `"moe"` | friend's | checkpoints-models + moe-checkpoint | 8-10 h |
| phase5-encdec | `"encdec"` | friend's (after moe) | checkpoints-models | 6-8 h |

Settings: GPU T4 x2, internet on, secrets GH_TOKEN (and HF_TOKEN) switched on.
Pipeline: mid-training on LSI text -> fine-tuning (full + LoRMA, 2 epochs, all chunks kept) -> whole-document Stage B ->
[KD: half-size student with / without the Stage B teacher] -> more techniques (1 epoch each, while time allows) ->
ensemble -> report + zip. The notebook stops starting new stages before Kaggle's 12-hour limit; to continue later,
attach this version's output as input and set RESUME = "auto" in cell 2.

In [ ]:
# 1. setup: code from GitHub
import os, subprocess, time
T0 = time.time()
from kaggle_secrets import UserSecretsClient
s = UserSecretsClient()
try:
    os.environ["HF_TOKEN"] = s.get_secret("HF_TOKEN")
except Exception:
    print("no HF_TOKEN secret: data download without it, no Hugging Face archive at the end")
GH = s.get_secret("GH_TOKEN")
REPO = "/kaggle/working/cs781-capstone"
cmds = ([["git", "clone", "-q", f"https://{GH}@github.com/NaveenOLA13/cs781-capstone.git", REPO]]
        if not os.path.isdir(REPO) else
        [["git", "-C", REPO, "fetch", "-q"], ["git", "-C", REPO, "reset", "-q", "--hard", "origin/main"]])
for c in cmds:
    if subprocess.run(c, capture_output=True).returncode != 0:
        raise RuntimeError("git failed: check the GH_TOKEN secret is attached to this notebook")
print(subprocess.run(["git", "-C", REPO, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# 2. what this notebook runs
MODEL = "llama"                         # "llama" | "decoder" | "moe" | "encdec"
RESUME = None                           # "auto" (or a folder holding lsi_runs/): continue an attached earlier version
BUDGET_H = 10.5                         # stop starting new stages after this many hours (Kaggle's limit is 12)

PLAN = {   # mid-training tokens (0 = none), main techniques (2 epochs, chunks kept for Stage B), other techniques (1 epoch)
    "llama":   dict(mid=40_000_000, main=("full", "lorma"), other=("adapter", "bitfit", "partial", "adalora",
                                                                       "unfreeze", "frozen"), epochs=2, batch=8),
    "decoder": dict(mid=40_000_000, main=("full", "lorma"), other=("adapter", "bitfit", "partial", "adalora",
                                                                       "unfreeze", "frozen"), epochs=2, batch=8),
    "moe":     dict(mid=20_000_000, main=("full", "lorma"), other=("adapter", "bitfit", "partial", "frozen"),
                    epochs=1, batch=4),
    "encdec":  dict(mid=0, main=("full", "bitfit"), other=("adapter", "partial", "frozen", "unfreeze", "lorma"),
                    epochs=2, batch=8),        # encdec: LoRMA / AdaLoRA on its RNN run without cuDNN (slow): last
}[MODEL]
KD_MODEL = "llama"                      # knowledge distillation is run in the llama notebook
COMMON = dict(train_chunks=3, eval_chunks=16, doc_tokens=8192, batch_size=PLAN["batch"],
              eval_batch_size=16 if MODEL == "moe" else 32)
OUT = "/kaggle/working/lsi_runs"
ENS = f"/kaggle/working/lsi_ensemble_{MODEL}"

import sys, glob, shutil
sys.path.insert(0, f"{REPO}/phase5")
HINTS = {"llama": ["llama"], "decoder": ["decoder"], "encdec": ["encdec", "encoder_decoder", "enc_dec"], "moe": ["moe"]}
def find_ckpt(model):
    c = set(glob.glob("/kaggle/input/**/*.pt", recursive=True))
    c |= {os.path.dirname(p) for p in glob.glob("/kaggle/input/**/data.pkl", recursive=True)}
    hits = []
    for p in sorted(c):
        low = p.lower().replace("-", "_")
        if "last" in os.path.basename(low) or "/mid5/" in low or "lsi_runs" in low:
            continue
        if model == "decoder" and any(h in low for h in HINTS["encdec"]):
            continue
        if any(h in low for h in HINTS[model]):
            hits.append(p)
    hits.sort(key=lambda p: (not p.endswith(".pt"), len(p)))
    return hits[0] if hits else None
CKPT = find_ckpt(MODEL)
print("Phase 1 checkpoint:", CKPT)
assert CKPT, "attach the checkpoints-models dataset (and moe-checkpoint for MoE) and wait until it is ready"
if RESUME == "auto":                    # the attached earlier version's output (the folder holding lsi_runs/)
    found = sorted(glob.glob("/kaggle/input/**/lsi_runs", recursive=True))
    RESUME = os.path.dirname(found[0]) if found else None
    print("resuming from", RESUME)
if RESUME:                              # continue an earlier version: copy its finished results back
    for sub in ("lsi_runs", "mid5", "lsi_cache"):
        src = os.path.join(RESUME, sub)
        if os.path.isdir(src) and not os.path.isdir(f"/kaggle/working/{sub}"):
            shutil.copytree(src, f"/kaggle/working/{sub}")
            print("resumed", src)
def hours():
    return (time.time() - T0) / 3600

In [ ]:
# 3. the data: sizes, statutes, and what the documents look like (for the report)
from lsi_data import load_data, summary, explore
data = load_data(log=print)
print(summary(data))
print(explore(data))

In [ ]:
# 4. self-tests on tiny stand-in models (~5 min); stops the notebook if something is broken
if not os.path.exists("/kaggle/working/tests_passed"):
    r = subprocess.run(f"cd {REPO}/phase5 && python test_lsi_data.py && python test_lsi_mid.py && "
                       f"python test_lsi_train.py && python test_lsi_hier.py", shell=True, capture_output=True, text=True)
    print(r.stdout[-4000:], r.stderr[-3000:])
    assert r.returncode == 0, "self-test failed: send the output above"
    open("/kaggle/working/tests_passed", "w").write("ok")
print(f"tests passed; {hours():.2f} h so far")

In [ ]:
# 5. mid-training: next-token prediction on the LSI training + dev documents (no labels, no test text)
if PLAN["mid"]:
    from lsi_mid import mid_train
    START = mid_train(MODEL, path=CKPT, tokens=PLAN["mid"], batch_size=8 if MODEL == "moe" else 16)
else:
    START = CKPT                         # the encoder-decoder is fine-tuned from its Phase 1 checkpoint
print("fine-tuning starts from", START, f"({hours():.2f} h so far)")

In [ ]:
# 6. main fine-tuning: two techniques at once (one per GPU), every chunk's vector and scores kept for Stage B
import json
from lsi_grid import run_grid, run_kd, ensemble, compare_all, report
run_grid(MODEL, techniques=PLAN["main"], ckpt=START, out_dir=OUT, epochs=PLAN["epochs"], save_chunks=True, **COMMON)
secs = [json.load(open(f"{OUT}/{MODEL}_{t}/metrics.json"))["train_seconds"] for t in PLAN["main"]
        if os.path.exists(f"{OUT}/{MODEL}_{t}/metrics.json")]
EPOCH_H = max(secs) / 3600 / PLAN["epochs"] if secs else 1.0      # one training epoch of the slower main run
print(f"one epoch takes about {EPOCH_H:.2f} h; {hours():.2f} h so far")

In [ ]:
# 7. whole-document reading (Stage B) on the main runs: statute-wise attention over all chunks of a document
from lsi_hier import run_stage_b
for tech in PLAN["main"]:
    run_dir = f"{OUT}/{MODEL}_{tech}"
    if os.path.exists(f"{run_dir}/chunks.pt") and not os.path.exists(f"{run_dir}/stageB/hier.txt"):
        run_stage_b(run_dir)
    elif os.path.exists(f"{run_dir}/stageB/hier.txt"):
        print(open(f"{run_dir}/stageB/hier.txt").read())
print(f"{hours():.2f} h so far")

In [ ]:
# 8. knowledge distillation (llama notebook): teacher = the best Stage B version of llama_full (reads the whole document);
#    student = the mid-trained llama with half of its blocks, trained with the teacher and, for comparison, without it
def best_stage_b(run_dir):
    best = None
    for v in glob.glob(f"{run_dir}/stageB/*/metrics.json"):
        m = json.load(open(v))
        cv = max(m["cv_global"], m["cv_per_label"])
        if best is None or cv > best[0]:
            best = (cv, os.path.dirname(v))
    return best[1] if best else run_dir
kd_h = 0.6 * EPOCH_H * PLAN["epochs"] + 0.4          # half-size students, both at once
if MODEL == KD_MODEL and hours() + kd_h + 0.5 < BUDGET_H:
    teacher = best_stage_b(f"{OUT}/{MODEL}_full")
    print("teacher:", teacher)
    run_kd(teacher, student_layers=0.5, out_dir=OUT)
else:
    print("knowledge distillation skipped here" + ("" if MODEL != KD_MODEL else " (not enough time left)"))
print(f"{hours():.2f} h so far")

In [ ]:
# 9. more techniques, two at a time, 1 epoch each, while time allows
others = list(PLAN["other"])
pair_h = EPOCH_H + 0.4                               # one epoch + tokenising and predicting dev / test
for i in range(0, len(others), 2):
    pair = tuple(others[i:i + 2])
    if hours() + pair_h + 0.5 > BUDGET_H:
        print(f"stopping before {pair}: {hours():.2f} h used, a pair takes about {pair_h:.2f} h")
        break
    t = time.time()
    run_grid(MODEL, techniques=pair, ckpt=START, out_dir=OUT, epochs=1, **COMMON)
    pair_h = max(pair_h, (time.time() - t) / 3600)
print(f"{hours():.2f} h so far")

In [ ]:
# 10. ensemble of this notebook's results (greedy on dev) + one table of everything
compare_all(OUT)
try:
    ens = ensemble([OUT], out_dir=ENS)
except Exception as e:
    print("ensemble skipped:", e)
report([OUT, ENS] if os.path.isdir(ENS) else [OUT], out_path="/kaggle/working/all_results.txt")

In [ ]:
# 11. results zip (no .pt files) + Hugging Face archive of the weights (skipped if the token cannot write)
import zipfile
from IPython.display import FileLink, display
name = f"lsi_results_{MODEL}.zip"
with zipfile.ZipFile(f"/kaggle/working/{name}", "w", zipfile.ZIP_DEFLATED) as z:
    for base in (OUT, ENS, "/kaggle/working/mid5"):
        for root, _, files in os.walk(base):
            for f in files:
                if not f.endswith(".pt"):
                    full = os.path.join(root, f)
                    z.write(full, os.path.relpath(full, "/kaggle/working"))
    z.write("/kaggle/working/all_results.txt", "all_results.txt")
display(FileLink(name))
try:
    from huggingface_hub import HfApi
    api = HfApi(token=os.environ["HF_TOKEN"])
    hf_repo = f"{api.whoami()['name']}/cs781-phase5-{MODEL}"
    api.create_repo(hf_repo, private=True, exist_ok=True)
    api.upload_folder(folder_path="/kaggle/working", repo_id=hf_repo, allow_patterns=["lsi_runs/**", "mid5/**"],
                      ignore_patterns=["**/chunks.pt", "**/_*"])
    print("weights archived on", hf_repo)
except Exception as e:
    print("Hugging Face upload skipped:", type(e).__name__, e)
print(f"done in {hours():.2f} h")